# Align calcium with 3D pose for one released session

This notebook uses the interpolation loader in `utlis/sync_utlis/loader_interp.py`:
- pose and COM are interpolated onto the miniscope timeline,
- calcium and dF/F are **not** interpolated,
- miniscope timestamps come from `miniscope/timeStamps.csv`,
- `metadata/frame_mapping.json` is used only for `time_offset` if `frametimes.npy` exists.

Sessions without miniscope (`oct3v1_beh_only`) are loaded with `load_behavior_only` at the end.


In [ ]:
import os
import json
import warnings

import numpy as np
import pandas as pd
import scipy.io as sio
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# Set RELEASE_ROOT to the downloaded release folder, and SESSION to cohort/kind/session.
RELEASE_ROOT = os.environ.get('BBOP_RELEASE_ROOT', '/path/to/release')
SESSION = 'oct3v1_mini/social/20241031_VC4+UNK1_S1'
rec_path = os.path.join(RELEASE_ROOT, SESSION)
pose_filename = 'pose3d_mouse22.mat'
com_filename = 'com3d.mat'
mapping_filename = 'frame_mapping.json'
kp_names = None
time_col = 'Time Stamp (ms)'
dff_percentile = 20
interp_kind = 'linear'
reference_h5_path = None  # Optional old aligned H5 for comparison


In [ ]:
from utlis.sync_utlis.loader_interp import merge_all_data_interp, load_behavior_only


In [ ]:
merged, beh_df, mini_df, pose_interp, ca_array, dff_array, merge_info = merge_all_data_interp(
    rec_path=rec_path,
    pose_filename=pose_filename,
    com_filename=com_filename,
    mapping_filename=mapping_filename,
    kp_names=kp_names,
    time_col=time_col,
    dff_percentile=dff_percentile,
    interp_kind=interp_kind,
)

print('merged shape:', merged.shape)
print('pose_interp shape:', pose_interp.shape)
print('ca shape:', ca_array.shape)
print('dff shape:', dff_array.shape)
print('merge_info:')
for k, v in merge_info.items():
    print(f'  {k}: {v}')
print()
print(merged.head())


In [ ]:
# Quick validation plot. If reference_h5_path is set, overlay the old result too.
t_sec = merged.index.to_numpy(dtype=float) / 1000.0
dff_cols = [c for c in merged.columns if c.startswith('dF_F_roi')]
calcium_cols = [c for c in merged.columns if c.startswith('calcium_roi')]
trace_col = dff_cols[0] if dff_cols else calcium_cols[0]
pose_cols = [c for c in merged.columns if c.endswith('_x_a1')]
pose_col = pose_cols[0] if pose_cols else None

ref = None
if reference_h5_path is not None:
    ref = pd.read_hdf(reference_h5_path)
    common_cols = [c for c in ['camera_frame_sixcam', trace_col, pose_col] if c is not None and c in ref.columns and c in merged.columns]
    print('Common comparison columns:', common_cols)
    for col in common_cols:
        n = min(len(merged), len(ref))
        a = merged[col].to_numpy()[:n]
        b = ref[col].to_numpy()[:n]
        mask = np.isfinite(a) & np.isfinite(b)
        max_abs_diff = np.max(np.abs(a[mask] - b[mask])) if np.any(mask) else np.nan
        print(f'{col}: max_abs_diff = {max_abs_diff}')

n_panels = 3 if pose_col is not None else 2
fig, axes = plt.subplots(n_panels, 1, figsize=(14, 2.8 * n_panels), sharex=True)
axes = np.atleast_1d(axes)

axes[0].plot(t_sec, merged['camera_frame_sixcam'].to_numpy(), lw=0.8, color='k', label='new')
if ref is not None and 'camera_frame_sixcam' in ref.columns:
    t_ref = ref.index.to_numpy(dtype=float) / 1000.0 if ref.index.name == 'timestamp_ms_mini' else np.arange(len(ref))
    axes[0].plot(t_ref, ref['camera_frame_sixcam'].to_numpy(), lw=0.8, alpha=0.7, label='reference')
axes[0].set_ylabel('sixcam frame')
axes[0].legend(loc='best')

axes[1].plot(t_sec, merged[trace_col].to_numpy(), lw=0.8, color='C0', label=f'new {trace_col}')
if ref is not None and trace_col in ref.columns:
    t_ref = ref.index.to_numpy(dtype=float) / 1000.0 if ref.index.name == 'timestamp_ms_mini' else np.arange(len(ref))
    axes[1].plot(t_ref, ref[trace_col].to_numpy(), lw=0.8, alpha=0.7, color='C1', label=f'reference {trace_col}')
axes[1].set_ylabel(trace_col)
axes[1].legend(loc='best')

if pose_col is not None:
    axes[2].plot(t_sec, merged[pose_col].to_numpy(), lw=0.8, color='C2', label=f'new {pose_col}')
    if ref is not None and pose_col in ref.columns:
        t_ref = ref.index.to_numpy(dtype=float) / 1000.0 if ref.index.name == 'timestamp_ms_mini' else np.arange(len(ref))
        axes[2].plot(t_ref, ref[pose_col].to_numpy(), lw=0.8, alpha=0.7, color='C3', label=f'reference {pose_col}')
    axes[2].set_ylabel(pose_col)
    axes[2].legend(loc='best')

axes[-1].set_xlabel('time (s)')
plt.tight_layout()
plt.show()


## Sessions without miniscope

`load_behavior_only` returns one row per pose frame on the camera clock (Camera1 `frametimes.npy`), with COM when the session has a `com3d.mat`.


In [ ]:
beh_session = os.path.join(RELEASE_ROOT, 'oct3v1_beh_only/single/20241004_VC11_S1')
beh_only_df, beh_pose, beh_com = load_behavior_only(beh_session)
print('pose:', beh_pose.shape, '| COM:', None if beh_com is None else beh_com.shape)
beh_only_df.head()


## Notes

- Release layout per session: `annotations/`, `metadata/`, `miniscope/`, `videos/`, `label3d_dannce.mat`.
- Pose arrays are `(frames, animals, 3, keypoints)` in millimetres; social sessions have 2 animals.
- It follows the interpolation path; pose and COM are interpolated onto miniscope timestamps, and calcium and dF/F are real values only.
